In [0]:
%sql
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # Manual validation checks
-- MAGIC
-- MAGIC This notebook checks that the Gold layer is correctly connected and that the cleaning rules worked.
-- MAGIC These are manual tests, which is acceptable for this project.

-- COMMAND ----------

-- 1) Silver and Gold fact table should have the same row count
SELECT
  (SELECT COUNT(*) FROM marathos.silver.races_obt) AS silver_rows,
  (SELECT COUNT(*) FROM marathos.gold.fct_results) AS fact_rows;

-- COMMAND ----------

-- 2) No missing event links
SELECT
  COUNT(*) AS missing_event_links
FROM marathos.gold.fct_results f
LEFT JOIN marathos.gold.dim_event e
  ON f.event_id = e.event_id
WHERE e.event_id IS NULL;

-- COMMAND ----------

-- 3) No missing athlete links
SELECT
  COUNT(*) AS missing_athlete_links
FROM marathos.gold.fct_results f
LEFT JOIN marathos.gold.dim_athlete a
  ON f.athlete_id = a.athlete_id
WHERE a.athlete_id IS NULL;

-- COMMAND ----------

-- 4) No missing date links
SELECT
  COUNT(*) AS missing_date_links
FROM marathos.gold.fct_results f
LEFT JOIN marathos.gold.dim_date d
  ON f.date_id = d.date_id
WHERE d.date_id IS NULL;

-- COMMAND ----------

-- 5) Speed should be inside the cleaning rule
SELECT
  MIN(speed_kmh) AS min_speed_kmh,
  MAX(speed_kmh) AS max_speed_kmh,
  AVG(speed_kmh) AS avg_speed_kmh
FROM marathos.gold.fct_results;

-- COMMAND ----------

-- 6) Count results by distance unit
SELECT
  distance_unit,
  COUNT(*) AS result_count
FROM marathos.gold.fct_results
GROUP BY distance_unit
ORDER BY result_count DESC;

-- COMMAND ----------

-- 7) Results by year
SELECT
  event_year,
  COUNT(*) AS result_count
FROM marathos.gold.fct_results
GROUP BY event_year
ORDER BY event_year;

-- COMMAND ----------

-- 8) Top athlete countries from km view
SELECT
  athlete_country,
  COUNT(*) AS result_count
FROM marathos.gold.v_marathon_km_results
GROUP BY athlete_country
ORDER BY result_count DESC
LIMIT 15;

-- COMMAND ----------

-- 9) Gender distribution
-- athlete_gender is in dim_athlete, not directly in fct_results
SELECT
  a.athlete_gender,
  COUNT(*) AS result_count
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a
  ON f.athlete_id = a.athlete_id
GROUP BY a.athlete_gender
ORDER BY result_count DESC;

-- COMMAND ----------

-- 10) Example analysis check: average finish time in hours for USA in 2018
SELECT
  a.athlete_country,
  e.event_year,
  ROUND(AVG(f.performance_seconds) / 3600, 2) AS avg_finish_hours
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a
  ON f.athlete_id = a.athlete_id
JOIN marathos.gold.dim_event e
  ON f.event_id = e.event_id
WHERE a.athlete_country = 'USA'
  AND e.event_year = 2018
GROUP BY a.athlete_country, e.event_year;

-- Final validation summary
-- This gives the most important checks in one visible table.

SELECT
  (SELECT COUNT(*)
   FROM marathos.silver.races_obt) AS silver_rows,

  (SELECT COUNT(*)
   FROM marathos.gold.fct_results) AS fact_rows,

  (SELECT COUNT(*)
   FROM marathos.gold.fct_results f
   LEFT JOIN marathos.gold.dim_event e
     ON f.event_id = e.event_id
   WHERE e.event_id IS NULL) AS missing_event_links,

  (SELECT COUNT(*)
   FROM marathos.gold.fct_results f
   LEFT JOIN marathos.gold.dim_athlete a
     ON f.athlete_id = a.athlete_id
   WHERE a.athlete_id IS NULL) AS missing_athlete_links,

  (SELECT COUNT(*)
   FROM marathos.gold.fct_results f
   LEFT JOIN marathos.gold.dim_date d
     ON f.date_id = d.date_id
   WHERE d.date_id IS NULL) AS missing_date_links,

  (SELECT MAX(speed_kmh)
   FROM marathos.gold.fct_results) AS max_speed_kmh;